# Klasifikasi Spasial Lahan Sawah dan Non-Sawah

Bagian ini mendokumentasikan tahapan akuisisi citra satelit optik **Sentinel-2A (Level-2A)** dalam format GeoTIFF (`.tif`) melalui **openEO**, ekstraksi nilai spektral pada **100 titik sampel** (50 sampel Sawah dan 50 sampel Non-Sawah) yang telah dilabeli melalui **QGIS**, hingga pemodelan klasifikasi biner (2 kelas).

## Instalasi Library Yang Digunakan

```bash
pip install geopandas
```

```bash
pip install openeo
```

```bash
pip install scikit-learn
```

## 1. Desain Pengambilan Sampel (Ground Truth)

Pengambilan sampel dilakukan secara spasial dengan membagi objek pengamatan ke dalam dua kelas seimbang (*balanced dataset*) dalam format vektor (`Shapefile` / `GeoJSON`):

| Kelas Target | Kode Label | Jumlah Sampel | Karakteristik Objek |
| :--- | :---: | :---: | :--- |
| **Sawah** | `1` | 50 Sampel | Petak lahan pertanian padi aktif (fase vegetatif, genangan air/tanam, maupun pematangan) |
| **Non-Sawah** | `0` | 50 Sampel | Permukiman/bangunan, jalan raya, badan air permanen, dan vegetasi non-pertanian |
| **Total** | ? | **100 Sampel** | Digabungkan menjadi satu *GeoDataFrame* berproyeksi `EPSG:4326` |



In [1]:
import geopandas as gpd
import openeo
import pandas as pd

# ==============================================================================
# 1. BACA KEDUA FILE ZIP (SAWAH & NON-SAWAH) DAN BERI LABEL OTOMATIS
# ==============================================================================
# Sesuaikan nama file zip non-sawah milikmu di baris kedua
gdf_sawah = gpd.read_file("sawah.zip").to_crs("EPSG:4326")
gdf_nonsawah = gpd.read_file("nonsawah.zip").to_crs("EPSG:4326")

# Beri kolom label secara otomatis
gdf_sawah["label_teks"] = "Sawah"
gdf_sawah["label"] = 1

gdf_nonsawah["label_teks"] = "Non-Sawah"
gdf_nonsawah["label"] = 0

# Gabungkan keduanya menjadi 1 GeoDataFrame (50 + 50 = 100 sampel)
gdf_gabungan = gpd.GeoDataFrame(
    pd.concat([gdf_sawah, gdf_nonsawah], ignore_index=True), crs="EPSG:4326"
)

print(f"Jumlah sampel Sawah     : {len(gdf_sawah)}")
print(f"Jumlah sampel Non-Sawah : {len(gdf_nonsawah)}")
print(f"Total sampel gabungan   : {len(gdf_gabungan)}")

# ==============================================================================
# 2. AMBIL BATAS KOORDINAT GABUNGAN (BOUNDING BOX) UNTUK OPENEO
# ==============================================================================
minx, miny, maxx, maxy = gdf_gabungan.total_bounds
buffer_deg = 0.005  # Tambahan margin ~500 meter agar titik di tepi tetap masuk

bbox = {
    "west": float(minx - buffer_deg),
    "south": float(miny - buffer_deg),
    "east": float(maxx + buffer_deg),
    "north": float(maxy + buffer_deg),
}
print("Bounding Box Gabungan:", bbox)

# ==============================================================================
# 3. KONEKSI KE OPENEO & UNDUH SENTINEL-2A FORMAT GEOTIFF (.tif)
# ==============================================================================
conn = openeo.connect("openeo.dataspace.copernicus.eu")
conn.authenticate_oidc()

datacube = conn.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=bbox,
    temporal_extent=["2026-05-01", "2026-09-30"],  # Rentang waktu minim awan
    bands=[
        "B02",
        "B03",
        "B04",
        "B08",
        "B11",
    ],  # Blue, Green, Red, NIR, SWIR
    max_cloud_cover=10,
)

# Ambil median waktu agar bebas tutupan awan
composite_s2 = datacube.median_time()

# Unduh menjadi file GeoTIFF (.tif)
nama_tif = "sentinel2_sawah_nonsawah.tif"
print("Mengunduh citra Sentinel-2A (.tif)...")
composite_s2.download(nama_tif, format="GTiff")
print(f"Berhasil diunduh: {nama_tif}")


Jumlah sampel Sawah     : 50
Jumlah sampel Non-Sawah : 50
Total sampel gabungan   : 100
Bounding Box Gabungan: {'west': 112.1375134, 'south': -7.0408626, 'east': 112.1617766, 'north': -7.0156369000000005}
Authenticated using refresh token.
Mengunduh citra Sentinel-2A (.tif)...
Berhasil diunduh: sentinel2_sawah_nonsawah.tif


**Output:**

```
Jumlah sampel Sawah     : 50
Jumlah sampel Non-Sawah : 50
Total sampel gabungan   : 100
Bounding Box Gabungan: {'west': 111.87169010000001, 'south': -6.9056517, 'east': 111.8930379, 'north': -6.8848906}
Authenticated using refresh token.
Mengunduh citra Sentinel-2A (.tif)...
Berhasil diunduh: sentinel2_sawah_nonsawah.tif
```

## 2. Akuisisi Citra Sentinel-2A (`.tif`) via openEO dan Ekstraksi Fitur

Akuisisi citra dilakukan menggunakan *bounding box* gabungan dari ke-100 titik sampel pada koleksi **`SENTINEL2_L2A`** (*Bottom-of-Atmosphere Reflectance*) dengan batas tutupan awan maksimum `< 10%` dan agregasi temporal `median_time()` untuk menghasilkan komposit citra bebas awan berformat **GeoTIFF (`.tif`)**.

### Fitur Spektral dan Indeks Turunan yang Diekstrak:
1. **Band Spektral Utama:**
   * `B02` (*Blue* - 490 nm), `B03` (*Green* - 560 nm), `B04` (*Red* - 665 nm) dengan resolusi spasial 10 meter.
   * `B08` (*Near Infrared / NIR* - 842 nm) untuk mendeteksi pantulan klorofil tanaman padi.
   * `B11` (*Short-Wave Infrared / SWIR* - 1610 nm) untuk mendeteksi kelembapan tanah dan genangan air.
2. **Normalized Difference Vegetation Index (NDVI):**

   $$\text{NDVI} = \frac{B08 - B04}{B08 + B04}$$

3. **Normalized Difference Water Index (NDWI):**

   $$\text{NDWI} = \frac{B03 - B08}{B03 + B08}$$



In [6]:
# Hapus data yang tidak memiliki geometri (None)
gdf_gabungan = gdf_gabungan[gdf_gabungan.geometry.notnull()]
gdf_gabungan = gdf_gabungan[~gdf_gabungan.is_empty]

# Lanjutkan ke proses selanjutnya
with rasterio.open("sentinel2_sawah_nonsawah.tif") as src:
  gdf_projected = gdf_gabungan.to_crs(src.crs)

  # Ambil centroid dari geometri yang valid
  koordinat_sampel = [
      (geom.centroid.x, geom.centroid.y) for geom in gdf_projected.geometry
  ]
  nilai_band = list(src.sample(koordinat_sampel))

In [15]:
import numpy as np
import rasterio
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# 1. Buka file .tif Sentinel-2A dan samakan proyeksi koordinat (CRS)
with rasterio.open("sentinel2_sawah_nonsawah.tif") as src:
  gdf_projected = gdf_gabungan.to_crs(src.crs)

  # Ambil titik tengah (centroid) dari tiap sampel (mendukung Point maupun Polygon)
  koordinat_sampel = [
      (geom.centroid.x, geom.centroid.y) for geom in gdf_projected.geometry
  ]

  # Ekstrak nilai piksel dari kelima band Sentinel-2A
  nilai_band = list(src.sample(koordinat_sampel))

# 2. Buat tabel DataFrame Fitur
df_dataset = pd.DataFrame(nilai_band, columns=["B02", "B03", "B04", "B08", "B11"])

# 3. Tambahkan fitur indeks vegetasi (NDVI) & indeks air (NDWI)
df_dataset["NDVI"] = (df_dataset["B08"] - df_dataset["B04"]) / (
    df_dataset["B08"] + df_dataset["B04"] + 1e-6
)
df_dataset["NDWI"] = (df_dataset["B03"] - df_dataset["B08"]) / (
    df_dataset["B03"] + df_dataset["B08"] + 1e-6
)

# 4. Masukkan Koordinat & Label Kelas (Sawah = 1, Non-Sawah = 0)
df_dataset["Longitude"] = gdf_gabungan.geometry.centroid.x
df_dataset["Latitude"] = gdf_gabungan.geometry.centroid.y
df_dataset["Kelas"] = gdf_gabungan["label_teks"]
df_dataset["Target"] = gdf_gabungan["label"]

# Simpan ke CSV (bisa dipakai juga kalau mau diolah di KNIME)
df_dataset.to_csv("dataset_100sampel_sawah_nonsawah.csv", index=False)
print("Dataset berhasil disimpan ke 'dataset_100sampel_sawah_nonsawah.csv'")
display(df_dataset.head())

# ==============================================================================
# 5. PROSES KLASIFIKASI 2 KELAS (SAWAH VS NON-SAWAH)
# ==============================================================================
fitur_kolom = ["B02", "B03", "B04", "B08", "B11", "NDVI", "NDWI"]
X = df_dataset[fitur_kolom]
y = df_dataset["Kelas"]

# Split 80% Training (40 Sawah + 40 Non-Sawah) & 20% Testing (10 Sawah + 10 Non-Sawah)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Latih model Random Forest
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)

# Evaluasi pada data uji
y_pred = model_rf.predict(X_test)
print(
    "\n=== HASIL EVALUASI KLASIFIKASI 2 KELAS ==="
)
print(f"Akurasi Testing : {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Dataset berhasil disimpan ke 'dataset_100sampel_sawah_nonsawah.csv'


C:\Users\ASUS\AppData\Local\Temp\ipykernel_15896\1953588402.py:31: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df_dataset["Longitude"] = gdf_gabungan.geometry.centroid.x
C:\Users\ASUS\AppData\Local\Temp\ipykernel_15896\1953588402.py:32: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df_dataset["Latitude"] = gdf_gabungan.geometry.centroid.y


,B02,B03,B04,B08,B11,NDVI,NDWI,Longitude,Latitude,Kelas,Target
0,304,567,291,3429,1434,0.843548,-0.716216,112.152833,-7.030260,Sawah,1.0
1,289,459,250,3153,1205,0.853071,-0.745847,112.154038,-7.029563,Sawah,1.0
2,277,506,324,3534,1524,0.832037,-0.749505,112.154886,-7.029610,Sawah,1.0
3,328,606,420,3124,1389,0.762980,-0.675067,112.153656,-7.030449,Sawah,1.0
4,349,612,360,2755,1378,0.768860,-0.636472,112.155967,-7.030698,Sawah,1.0


ValueError: Input contains NaN

In [4]:
pip install rasterio

   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/30.7 MB ? eta -:--:--
   - -------------------------------------- 0.8/30.7 MB 1.4 MB/s eta 0:00:23
   - -------------------------------------- 0.8/30.7 MB 1.4 MB/s eta 0:00:23
   - -------------------------------------- 0.8/30.7 MB 1.4 MB/s eta 0:00:23
   - -------------------------------------- 1.0/30.7 MB 730.9 kB/s eta 0:00:41
   - -------------------------------------- 1.0/30.7 MB 730.9 kB/s eta 0:00:41
   - -------------------------------------- 1.0/30.7 MB 730.9 kB/s eta 0:00:41
   - -------------------------------------- 1.0/30.7 MB 730.9 kB/s eta 0:00:41
   - -------------------------------------- 1.0/30.7 MB 730.9 kB/s eta 0:00:41
   - -------------------------------------- 1.3/30.7 MB 529.0 kB/s eta 0:00:56
   -- ---------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# 1. Muat dataset dari CSV
df_dataset = pd.read_csv("../../Data/sawah/dataset_100sampel_sawah_nonsawah.csv")

# 2. Hapus semua baris yang mengandung NaN
df_dataset = df_dataset.dropna(subset=["Kelas"]) # Atau gunakan df_dataset.dropna() untuk semua kolom

# 3. Siapkan fitur dan target
fitur_kolom = [c for c in ["B02", "B03", "B04", "B08", "B11", "NDVI", "NDWI"] if c in df_dataset.columns]
X = df_dataset[fitur_kolom]
y = df_dataset["Kelas"]

# 4. Split 80% Training & 20% Testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [16]:
print(f"Jumlah sampel Training: {len(X_train)}")

Jumlah sampel Training: 76


Jumlah sampel Training: 76


In [18]:
pip install folium


   ------------- -------------------------- 1/3 [branca]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   ---------------------------------------- 3/3 [folium]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import folium
from IPython.display import display

# Pastikan dataset memiliki label dan koordinat geografis.
kolom_peta = ["Longitude", "Latitude", "Kelas"]
kolom_tidak_ada = [kolom for kolom in kolom_peta if kolom not in df_dataset.columns]
if kolom_tidak_ada:
    raise ValueError(f"Kolom untuk peta tidak ditemukan: {kolom_tidak_ada}")

df_peta = df_dataset[kolom_peta].copy()
df_peta["Longitude"] = pd.to_numeric(df_peta["Longitude"], errors="coerce")
df_peta["Latitude"] = pd.to_numeric(df_peta["Latitude"], errors="coerce")
df_peta = df_peta.dropna(subset=kolom_peta)
df_peta = df_peta[
    np.isfinite(df_peta["Longitude"])
    & np.isfinite(df_peta["Latitude"])
    & df_peta["Longitude"].between(-180, 180)
    & df_peta["Latitude"].between(-90, 90)
]

warna_kelas = {"Sawah": "green", "Non-Sawah": "red"}
kelas_tidak_dikenal = sorted(set(df_peta["Kelas"]) - set(warna_kelas))
if kelas_tidak_dikenal:
    raise ValueError(f"Label kelas tidak dikenali: {kelas_tidak_dikenal}")
if df_peta.empty:
    raise ValueError("Tidak ada baris dengan koordinat valid untuk ditampilkan pada peta.")

peta = folium.Map(
    location=[df_peta["Latitude"].mean(), df_peta["Longitude"].mean()],
    zoom_start=14,
    tiles="OpenStreetMap",
)

for kelas, warna in warna_kelas.items():
    layer = folium.FeatureGroup(name=kelas, show=True)
    data_kelas = df_peta[df_peta["Kelas"] == kelas]
    for _, sampel in data_kelas.iterrows():
        folium.CircleMarker(
            location=[sampel["Latitude"], sampel["Longitude"]],
            radius=6,
            color=warna,
            fill=True,
            fill_color=warna,
            fill_opacity=0.8,
            tooltip=kelas,
        ).add_to(layer)
    layer.add_to(peta)

folium.LayerControl().add_to(peta)
peta.save("peta_sampel_sawah_nonsawah.html")
display(peta)